## Imports

In [1]:
from pathlib import Path

import cdsapi

## Locate the project and output folder

In [2]:
project_root = Path.cwd().resolve()

while project_root != project_root.parent:
    if (project_root / "data").exists():
        break

    project_root = project_root.parent

weather_folder = (
    project_root
    / "data"
    / "raw"
    / "weather"
    / "era5_land"
    / "2024"
)

weather_folder.mkdir(
    parents=True,
    exist_ok=True,
)

july_path = (
    weather_folder
    / "era5_land_manitoba_2024_07_hourly.nc"
)

august_buffer_path = (
    weather_folder
    / "era5_land_manitoba_2024_08_01_hourly.nc"
)

print("Project root:", project_root)
print("Weather folder:", weather_folder)
print("July output:", july_path)
print("August buffer output:", august_buffer_path)

Project root: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence
Weather folder: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/raw/weather/era5_land/2024
July output: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/raw/weather/era5_land/2024/era5_land_manitoba_2024_07_hourly.nc
August buffer output: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/raw/weather/era5_land/2024/era5_land_manitoba_2024_08_01_hourly.nc


## Define the request settings

In [3]:
DATASET_NAME = "reanalysis-era5-land"

VARIABLES = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "total_precipitation",
]

TIMES = [
    f"{hour:02d}:00"
    for hour in range(24)
]

MANITOBA_AREA = [
    60.2,     # North
    -102.3,   # West
    48.8,     # South
    -89.5,    # East
]

client = cdsapi.Client()

print("Number of hourly timestamps per day:", len(TIMES))

Number of hourly timestamps per day: 24


## Create a reusable download function

In [4]:
def download_era5_land(
    *,
    year: str,
    month: str,
    days: list[str],
    output_path: Path,
) -> None:
    """Download ERA5-Land hourly weather for Manitoba."""

    if output_path.exists():
        size_mb = output_path.stat().st_size / 1_000_000

        print(
            f"File already exists: {output_path.name} "
            f"({size_mb:.2f} MB)"
        )

        return

    request = {
        "variable": VARIABLES,
        "year": year,
        "month": month,
        "day": days,
        "time": TIMES,
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": MANITOBA_AREA,
    }

    print(
        f"Requesting {year}-{month}: "
        f"{len(days)} day(s)"
    )

    client.retrieve(
        DATASET_NAME,
        request,
        str(output_path),
    )

    size_mb = output_path.stat().st_size / 1_000_000

    print(
        f"Downloaded: {output_path.name} "
        f"({size_mb:.2f} MB)"
    )

## Download July 1–31

In [5]:
july_days = [
    f"{day:02d}"
    for day in range(1, 32)
]

download_era5_land(
    year="2024",
    month="07",
    days=july_days,
    output_path=july_path,
)

Requesting 2024-07: 31 day(s)


2026-07-23 11:44:11,475 INFO Request ID is ffcc5d82-5fbb-4f1f-8e2b-57dd48c12791
2026-07-23 11:44:11,660 INFO status has been updated to accepted
2026-07-23 11:44:33,564 INFO status has been updated to running
2026-07-23 11:50:35,056 INFO status has been updated to successful
                                                                                         

Downloaded: era5_land_manitoba_2024_07_hourly.nc (93.53 MB)


## Download the August 1 buffer

In [6]:
download_era5_land(
    year="2024",
    month="08",
    days=["01"],
    output_path=august_buffer_path,
)

Requesting 2024-08: 1 day(s)


2026-07-23 11:50:57,800 INFO Request ID is 9806b41c-f65f-47c1-90eb-c38bf6cbd2c5
2026-07-23 11:50:57,988 INFO status has been updated to accepted
2026-07-23 11:51:12,058 INFO status has been updated to running
2026-07-23 11:51:31,383 INFO status has been updated to successful
                                                                                         

Downloaded: era5_land_manitoba_2024_08_01_hourly.nc (2.67 MB)


## Verify both files

In [7]:
for path in [
    july_path,
    august_buffer_path,
]:
    print("\nFile:", path.name)
    print("Exists:", path.exists())

    if path.exists():
        print(
            "Size:",
            round(
                path.stat().st_size / 1_000_000,
                2,
            ),
            "MB",
        )


File: era5_land_manitoba_2024_07_hourly.nc
Exists: True
Size: 93.53 MB

File: era5_land_manitoba_2024_08_01_hourly.nc
Exists: True
Size: 2.67 MB


## Additional imports

In [8]:
import numpy as np
import pandas as pd
import xarray as xr

## Open and combine both hourly files

In [9]:
hourly_weather_raw = xr.open_mfdataset(
    [
        july_path,
        august_buffer_path,
    ],
    combine="by_coords",
    chunks={"valid_time": 168},
)

hourly_weather_raw = hourly_weather_raw.sortby(
    "valid_time"
)

time_index = pd.DatetimeIndex(
    hourly_weather_raw["valid_time"].values
)

print("Hourly dimensions:")
print(dict(hourly_weather_raw.sizes))

print("\nFirst timestamp:", time_index.min())
print("Last timestamp:", time_index.max())
print("Hourly timestamps:", len(time_index))
print("Duplicate timestamps:", time_index.duplicated().sum())

print("\nVariables:")
print(list(hourly_weather_raw.data_vars))

Hourly dimensions:
{'valid_time': 768, 'latitude': 115, 'longitude': 129}

First timestamp: 2024-07-01 00:00:00
Last timestamp: 2024-08-01 23:00:00
Hourly timestamps: 768
Duplicate timestamps: 0

Variables:
['t2m', 'd2m', 'u10', 'v10', 'tp']


/var/folders/dk/g6kdw5nj5hb8tlfg5r6rfn300000gn/T/ipykernel_22389/127936601.py:1: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 168. This could degrade performance. Instead, consider rechunking after loading.
  hourly_weather_raw = xr.open_mfdataset(


## Create hourly weather variables

In [10]:
hourly_weather = xr.Dataset(
    {
        "TEMPERATURE_C": (
            hourly_weather_raw["t2m"] - 273.15
        ),
        "DEWPOINT_C": (
            hourly_weather_raw["d2m"] - 273.15
        ),
        "WIND_SPEED_MS": np.sqrt(
            hourly_weather_raw["u10"] ** 2
            + hourly_weather_raw["v10"] ** 2
        ),
        "PRECIPITATION_MM": (
            hourly_weather_raw["tp"] * 1000
        ),
    }
)

hourly_weather["RELATIVE_HUMIDITY_PCT"] = (
    100
    * np.exp(
        (
            17.625 * hourly_weather["DEWPOINT_C"]
            / (243.04 + hourly_weather["DEWPOINT_C"])
        )
        -
        (
            17.625 * hourly_weather["TEMPERATURE_C"]
            / (243.04 + hourly_weather["TEMPERATURE_C"])
        )
    )
).clip(min=0, max=100)

print(list(hourly_weather.data_vars))

['TEMPERATURE_C', 'DEWPOINT_C', 'WIND_SPEED_MS', 'PRECIPITATION_MM', 'RELATIVE_HUMIDITY_PCT']


## Convert UTC timestamps to Manitoba dates

In [11]:
utc_times = pd.DatetimeIndex(
    hourly_weather["valid_time"].values
).tz_localize("UTC")

local_times = utc_times.tz_convert(
    "America/Winnipeg"
)

instantaneous_dates = (
    local_times
    .tz_localize(None)
    .normalize()
    .to_numpy(dtype="datetime64[ns]")
)

# Precipitation belongs to the interval ending at each timestamp.
precipitation_dates = (
    (local_times - pd.Timedelta(nanoseconds=1))
    .tz_localize(None)
    .normalize()
    .to_numpy(dtype="datetime64[ns]")
)

print("First UTC time:", utc_times[0])
print("First Manitoba time:", local_times[0])
print("Last Manitoba time:", local_times[-1])

First UTC time: 2024-07-01 00:00:00+00:00
First Manitoba time: 2024-06-30 19:00:00-05:00
Last Manitoba time: 2024-08-01 18:00:00-05:00


## Aggregate hourly data to daily weather

In [12]:
hourly_weather = hourly_weather.assign_coords(
    LOCAL_DATE=(
        "valid_time",
        instantaneous_dates,
    )
)

precipitation_hourly = (
    hourly_weather["PRECIPITATION_MM"]
    .assign_coords(
        PRECIPITATION_DATE=(
            "valid_time",
            precipitation_dates,
        )
    )
)

daily_precipitation = (
    precipitation_hourly
    .groupby("PRECIPITATION_DATE")
    .sum("valid_time")
    .rename(
        {"PRECIPITATION_DATE": "LOCAL_DATE"}
    )
)

daily_weather = xr.Dataset(
    {
        "TEMP_MAX_C": (
            hourly_weather["TEMPERATURE_C"]
            .groupby("LOCAL_DATE")
            .max("valid_time")
        ),
        "TEMP_MIN_C": (
            hourly_weather["TEMPERATURE_C"]
            .groupby("LOCAL_DATE")
            .min("valid_time")
        ),
        "TEMP_MEAN_C": (
            hourly_weather["TEMPERATURE_C"]
            .groupby("LOCAL_DATE")
            .mean("valid_time")
        ),
        "RH_MIN_PCT": (
            hourly_weather["RELATIVE_HUMIDITY_PCT"]
            .groupby("LOCAL_DATE")
            .min("valid_time")
        ),
        "RH_MEAN_PCT": (
            hourly_weather["RELATIVE_HUMIDITY_PCT"]
            .groupby("LOCAL_DATE")
            .mean("valid_time")
        ),
        "WIND_MAX_MS": (
            hourly_weather["WIND_SPEED_MS"]
            .groupby("LOCAL_DATE")
            .max("valid_time")
        ),
        "WIND_MEAN_MS": (
            hourly_weather["WIND_SPEED_MS"]
            .groupby("LOCAL_DATE")
            .mean("valid_time")
        ),
        "PRECIPITATION_MM": daily_precipitation,
    }
)

print(dict(daily_weather.sizes))

{'latitude': 115, 'longitude': 129, 'LOCAL_DATE': 33}


## Verify all July dates are complete

In [14]:
instantaneous_counts = pd.Series(
    instantaneous_dates
).value_counts()

precipitation_counts = pd.Series(
    precipitation_dates
).value_counts()

july_dates = pd.date_range(
    "2024-07-01",
    "2024-07-31",
    freq="D",
)

incomplete_dates = []

for date in july_dates:
    instantaneous_hours = instantaneous_counts.get(
        date,
        0,
    )

    precipitation_hours = precipitation_counts.get(
        date,
        0,
    )

    if (
        instantaneous_hours != 24
        or precipitation_hours != 24
    ):
        incomplete_dates.append(
            {
                "date": date,
                "instantaneous_hours": instantaneous_hours,
                "precipitation_hours": precipitation_hours,
            }
        )

print("Expected July dates:", len(july_dates))
print("Incomplete July dates:", len(incomplete_dates))

if incomplete_dates:
    display(pd.DataFrame(incomplete_dates))

Expected July dates: 31
Incomplete July dates: 0


## Keep July 1–31 only

In [15]:
july_date_values = july_dates.to_numpy(
    dtype="datetime64[ns]"
)

daily_weather_july = daily_weather.sel(
    LOCAL_DATE=july_date_values
).load()

# The daily data are now loaded, so the hourly files can be closed.
hourly_weather_raw.close()

print("Daily July dimensions:")
print(dict(daily_weather_july.sizes))

print("\nFirst date:")
print(daily_weather_july["LOCAL_DATE"].values[0])

print("\nLast date:")
print(daily_weather_july["LOCAL_DATE"].values[-1])

Daily July dimensions:
{'latitude': 115, 'longitude': 129, 'LOCAL_DATE': 31}

First date:
2024-07-01T00:00:00.000000000

Last date:
2024-07-31T00:00:00.000000000


## Save the daily gridded weather

In [16]:
daily_weather_path = (
    weather_folder
    / "era5_land_manitoba_2024_07_daily.nc"
)

daily_weather_july.to_netcdf(
    daily_weather_path
)

print("Daily weather saved:", daily_weather_path)
print("File exists:", daily_weather_path.exists())

if daily_weather_path.exists():
    print(
        "File size:",
        round(
            daily_weather_path.stat().st_size
            / 1_000_000,
            2,
        ),
        "MB",
    )

Daily weather saved: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/raw/weather/era5_land/2024/era5_land_manitoba_2024_07_daily.nc
File exists: True
File size: 14.78 MB


## Load the saved ERA5 matching table

In [17]:
era5_match_path = (
    project_root
    / "data"
    / "interim"
    / "manitoba_grid_to_era5_land_matches.parquet"
)

era5_matches = pd.read_parquet(
    era5_match_path
)

print("ERA5 match rows:", len(era5_matches))
print(
    "Duplicate GRID_ID values:",
    era5_matches["GRID_ID"].duplicated().sum(),
)

display(era5_matches.head())

ERA5 match rows: 6501
Duplicate GRID_ID values: 0


,GRID_ID,CENTER_LATITUDE,CENTER_LONGITUDE,MB_AREA_KM2,MB_COVERAGE_PCT,ERA5_LATITUDE,ERA5_LONGITUDE,ERA5_MATCH_DISTANCE_KM,ERA5_MATCH_QUALITY
0,MB_005_000,49.034036,-101.408339,14.489028,14.489028,49.0,-101.4,3.833177,Near: 0–10 km
1,MB_006_000,49.122970,-101.428824,2.229798,2.229798,49.1,-101.4,3.305338,Near: 0–10 km
2,MB_007_000,49.211928,-101.449396,7.044500,7.044500,49.2,-101.4,3.825834,Near: 0–10 km
3,MB_008_000,49.300912,-101.470056,0.002842,0.002842,49.3,-101.5,2.173562,Near: 0–10 km
4,MB_004_001,48.958440,-101.252939,4.265625,4.265625,49.0,-101.3,5.757781,Near: 0–10 km


## Assign July weather to all Manitoba cells

In [18]:
grid_ids = era5_matches["GRID_ID"].to_numpy()

latitude_indexer = xr.DataArray(
    era5_matches["ERA5_LATITUDE"].to_numpy(),
    dims="GRID_ID",
    coords={"GRID_ID": grid_ids},
)

longitude_indexer = xr.DataArray(
    era5_matches["ERA5_LONGITUDE"].to_numpy(),
    dims="GRID_ID",
    coords={"GRID_ID": grid_ids},
)

weather_at_grid = daily_weather_july.sel(
    latitude=latitude_indexer,
    longitude=longitude_indexer,
    method="nearest",
)

print("Weather-at-grid dimensions:")
print(dict(weather_at_grid.sizes))

Weather-at-grid dimensions:
{'GRID_ID': 6501, 'LOCAL_DATE': 31}


## Convert July weather to a table

In [19]:
weather_grid_df = (
    weather_at_grid
    .to_dataframe()
    .reset_index()
    .rename(
        columns={
            "latitude": "ERA5_LATITUDE",
            "longitude": "ERA5_LONGITUDE",
        }
    )
)

weather_grid_df["LOCAL_DATE"] = (
    pd.to_datetime(weather_grid_df["LOCAL_DATE"])
    .dt.normalize()
)

match_metadata = [
    "GRID_ID",
    "CENTER_LATITUDE",
    "CENTER_LONGITUDE",
    "MB_AREA_KM2",
    "MB_COVERAGE_PCT",
    "ERA5_MATCH_DISTANCE_KM",
    "ERA5_MATCH_QUALITY",
]

weather_grid_df = weather_grid_df.merge(
    era5_matches[match_metadata],
    on="GRID_ID",
    how="left",
    validate="many_to_one",
)

print("Expected rows:", 31 * 6501)
print("Actual rows:", len(weather_grid_df))

print(
    "Duplicate date-grid rows:",
    weather_grid_df.duplicated(
        subset=["LOCAL_DATE", "GRID_ID"]
    ).sum(),
)

Expected rows: 201531
Actual rows: 201531
Duplicate date-grid rows: 0


## Validate weather completeness

In [20]:
weather_variables = [
    "TEMP_MAX_C",
    "TEMP_MIN_C",
    "TEMP_MEAN_C",
    "RH_MIN_PCT",
    "RH_MEAN_PCT",
    "WIND_MAX_MS",
    "WIND_MEAN_MS",
    "PRECIPITATION_MM",
]

print("Missing weather values:")

display(
    weather_grid_df[
        weather_variables
    ]
    .isna()
    .sum()
    .to_frame("missing_count")
)

print("\nWeather summaries:")

display(
    weather_grid_df[
        weather_variables
    ]
    .describe()
    .T
)

Missing weather values:


,missing_count
TEMP_MAX_C,0
TEMP_MIN_C,0
TEMP_MEAN_C,0
RH_MIN_PCT,0
RH_MEAN_PCT,0
WIND_MAX_MS,0
WIND_MEAN_MS,0
PRECIPITATION_MM,0



Weather summaries:


,count,mean,std,min,25%,50%,75%,max
TEMP_MAX_C,201531.0,23.394640,4.212975,3.658112,20.775543,23.780426,26.455231,33.324371
TEMP_MIN_C,201531.0,14.881010,3.651887,-0.204437,12.768463,15.053864,17.398346,25.239899
TEMP_MEAN_C,201531.0,19.121708,3.563828,2.447021,16.934576,19.534140,21.715042,27.127945
RH_MIN_PCT,201531.0,52.907822,12.236391,23.715736,43.441923,52.276611,60.910957,99.397209
RH_MEAN_PCT,201531.0,72.112335,8.783276,39.935337,65.930328,72.049034,77.986195,99.959595
WIND_MAX_MS,201531.0,4.196836,1.666722,1.314093,3.007422,3.746355,5.023724,13.370600
WIND_MEAN_MS,201531.0,2.898803,1.235708,0.583286,2.054993,2.586883,3.453409,10.398984
PRECIPITATION_MM,201531.0,28.274815,58.741886,0.000000,0.690343,6.269763,27.194252,723.971069


## Add the wildfire targets

In [21]:
targets_path = (
    project_root
    / "data"
    / "interim"
    / "manitoba_daily_fire_targets_2005_2025.parquet"
)

fire_targets = pd.read_parquet(
    targets_path
)

fire_targets["FIRE_DATE"] = (
    pd.to_datetime(fire_targets["FIRE_DATE"])
    .dt.normalize()
)

target_fields = [
    "FIRE_DATE",
    "GRID_ID",
    "FIRE_OCCURRED",
    "FIRE_COUNT",
    "NATURAL_FIRE_COUNT",
    "HUMAN_FIRE_COUNT",
    "UNKNOWN_FIRE_COUNT",
    "TOTAL_BURNED_HA",
    "MAX_FIRE_SIZE_HA",
]

july_model_table = weather_grid_df.merge(
    fire_targets[target_fields],
    left_on=["LOCAL_DATE", "GRID_ID"],
    right_on=["FIRE_DATE", "GRID_ID"],
    how="left",
    validate="one_to_one",
)

count_columns = [
    "FIRE_OCCURRED",
    "FIRE_COUNT",
    "NATURAL_FIRE_COUNT",
    "HUMAN_FIRE_COUNT",
    "UNKNOWN_FIRE_COUNT",
]

july_model_table[count_columns] = (
    july_model_table[count_columns]
    .fillna(0)
    .astype("int16")
)

size_columns = [
    "TOTAL_BURNED_HA",
    "MAX_FIRE_SIZE_HA",
]

july_model_table[size_columns] = (
    july_model_table[size_columns]
    .fillna(0.0)
)

july_model_table = july_model_table.drop(
    columns="FIRE_DATE"
)

print("Model-table rows:", len(july_model_table))

print(
    "Positive grid-cell days:",
    july_model_table["FIRE_OCCURRED"].sum(),
)

print(
    "Fires represented:",
    july_model_table["FIRE_COUNT"].sum(),
)

print(
    "Duplicate date-grid rows:",
    july_model_table.duplicated(
        subset=["LOCAL_DATE", "GRID_ID"]
    ).sum(),
)

Model-table rows: 201531
Positive grid-cell days: 111
Fires represented: 118
Duplicate date-grid rows: 0


## Save July tables

In [22]:
weather_output_path = (
    project_root
    / "data"
    / "interim"
    / "era5_grid_weather_2024_07.parquet"
)

model_output_path = (
    project_root
    / "data"
    / "interim"
    / "manitoba_model_table_2024_07.parquet"
)

weather_grid_df.to_parquet(
    weather_output_path,
    index=False,
)

july_model_table.to_parquet(
    model_output_path,
    index=False,
)

print("Weather table saved:", weather_output_path)
print("Weather file exists:", weather_output_path.exists())

print("\nModel table saved:", model_output_path)
print("Model file exists:", model_output_path.exists())

Weather table saved: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/interim/era5_grid_weather_2024_07.parquet
Weather file exists: True

Model table saved: /Users/eleazar/Documents/manitoba-wildfire-risk-intelligence/data/interim/manitoba_model_table_2024_07.parquet
Model file exists: True


## Final July validation

In [23]:
july_target_check = fire_targets.loc[
    fire_targets["FIRE_DATE"].between(
        "2024-07-01",
        "2024-07-31",
    )
].copy()

print("Target-table positive rows:", len(july_target_check))
print("Target-table fires:", july_target_check["FIRE_COUNT"].sum())

print(
    "Model-table positive rows:",
    july_model_table["FIRE_OCCURRED"].sum(),
)

print(
    "Model-table fires:",
    july_model_table["FIRE_COUNT"].sum(),
)

assert len(july_target_check) == 111
assert july_target_check["FIRE_COUNT"].sum() == 118
assert july_model_table["FIRE_OCCURRED"].sum() == 111
assert july_model_table["FIRE_COUNT"].sum() == 118

print("\nJuly target validation passed.")

Target-table positive rows: 111
Target-table fires: 118
Model-table positive rows: 111
Model-table fires: 118

July target validation passed.
